# Vaja 5 – Multipla regresija in interpretacija modelov

**Cilj:** Razumeti, kako regresijski modeli povezujejo več vhodnih spremenljivk z izhodno spremenljivko ter interpretirati rezultate.

**Proces:** Analiza 3D tiskanja medicinskih implantatov

**Faze:** Analyze in Improve (Lean Six Sigma)

## 0️⃣ Uvoz knjižnic

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Za regresijske modele
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, classification_report, confusion_matrix, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import statsmodels.api as sm
from statsmodels.formula.api import ols, logit
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

# Nastavitve prikaza
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Knjižnice uspešno naložene")

## 1️⃣ Priprava podatkov

### 1.1 Nalaganje podatkov

In [ ]:
# Naložimo podatke
df = pd.read_csv('expanded_data.csv')

print(f"Dimenzije podatkov: {df.shape}")
print(f"\nPrvih 5 vrstic:")
df.head()

In [ ]:
# Pregled strukture podatkov
print("Informacije o podatkih:")
print(df.info())
print("\n" + "="*50)
print("Opisne statistike:")
df.describe()

In [ ]:
# Preveri manjkajoče vrednosti
print("Manjkajoče vrednosti:")
print(df.isnull().sum())

### 1.2 Opis spremenljivk

**Obstoječe spremenljivke:**

| Spremenljivka | Tip | Opis |
|---------------|-----|------|
| `Iteracija` | Numerična | Številka iteracije procesa (1-10) |
| `Korak` | Kategorična | Ime koraka v procesu 3D tiskanja |
| `Vloga (kdo)` | Kategorična | Oseba/naprava, ki izvaja korak |
| `Trajanje (h)` | Numerična | **Čas trajanja koraka v urah** |
| `Cakanje (h)` | Numerična | Čas čakanja pred korakom v urah |
| `Tip aktivnosti (VA/NVA)` | Kategorična | VA = Value Added, NVA = Non-Value Added, NNVA = Necessary Non-Value Added |
| `Napake (st)` | Numerična | Število napak v tem koraku |
| `Trajanje_kategorija` | Ordinalna | Kategorija trajanja (Kratko, Srednje, Dolgo) |
| `Cakanje_intenziteta` | Ordinalna | Intenziteta čakanja (Brez, Zmerno, Dolgo) |

### 1.3 Kreiranje novih spremenljivk

In [ ]:
# Kreirajmo nove koristne spremenljivke za analizo
df_model = df.copy()

# 1. Binarna spremenljivka za napake (Ali je prišlo do napake?)
df_model['Ima_napako'] = (df_model['Napake (st)'] > 0).astype(int)

# 2. Skupni čas (Trajanje + Čakanje)
df_model['Skupni_cas'] = df_model['Trajanje (h)'] + df_model['Cakanje (h)']

# 3. Razmerje čakanja glede na trajanje
df_model['Razmerje_cakanje_trajanje'] = df_model['Cakanje (h)'] / (df_model['Trajanje (h)'] + 0.01)

# 4. Binarna spremenljivka za tip aktivnosti
df_model['Je_VA'] = (df_model['Tip aktivnosti (VA/NVA)'] == 'VA').astype(int)
df_model['Je_NVA'] = (df_model['Tip aktivnosti (VA/NVA)'] == 'NVA').astype(int)

# 5. Kodiranje kategoričnih spremenljivk
le_korak = LabelEncoder()
df_model['Korak_code'] = le_korak.fit_transform(df_model['Korak'])

le_vloga = LabelEncoder()
df_model['Vloga_code'] = le_vloga.fit_transform(df_model['Vloga (kdo)'])

# 6. Ordinalne spremenljivke - numerično kodiranje
trajanje_map = {'Kratko': 0, 'Srednje': 1, 'Dolgo': 2}
cakanje_map = {'Brez': 0, 'Zmerno': 1, 'Dolgo': 2}

df_model['Trajanje_ord'] = df_model['Trajanje_kategorija'].map(trajanje_map)
df_model['Cakanje_ord'] = df_model['Cakanje_intenziteta'].map(cakanje_map)

# 7. Standardizirane spremenljivke za numerične vrednosti
scaler = StandardScaler()
df_model['Trajanje_std'] = scaler.fit_transform(df_model[['Trajanje (h)']])
df_model['Cakanje_std'] = scaler.fit_transform(df_model[['Cakanje (h)']])

print("✅ Nove spremenljivke uspešno kreirane")
print(f"\nŠtevilo spremenljivk: {df_model.shape[1]}")
print(f"\nNove spremenljivke:")
print(df_model[['Trajanje (h)', 'Cakanje (h)', 'Ima_napako', 'Skupni_cas', 
                'Razmerje_cakanje_trajanje', 'Je_VA', 'Je_NVA', 
                'Trajanje_ord', 'Cakanje_ord']].head(10))

### 1.4 Vizualizacija podatkov

In [ ]:
# Korelacijska matrika
fig, ax = plt.subplots(figsize=(12, 10))
numeric_cols = ['Trajanje (h)', 'Cakanje (h)', 'Napake (st)', 'Skupni_cas', 
                'Razmerje_cakanje_trajanje', 'Je_VA', 'Je_NVA', 'Iteracija',
                'Trajanje_ord', 'Cakanje_ord']
correlation = df_model[numeric_cols].corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.title('Korelacijska matrika spremenljivk', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribucija ciljnih spremenljivk
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Trajanje
axes[0, 0].hist(df_model['Trajanje (h)'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribucija: Trajanje (h)', fontweight='bold')
axes[0, 0].set_xlabel('Trajanje (h)')
axes[0, 0].set_ylabel('Frekvenca')

# Čakanje
axes[0, 1].hist(df_model['Cakanje (h)'], bins=20, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Distribucija: Čakanje (h)', fontweight='bold')
axes[0, 1].set_xlabel('Čakanje (h)')
axes[0, 1].set_ylabel('Frekvenca')

# Napake
napake_counts = df_model['Ima_napako'].value_counts().sort_index()
axes[1, 0].bar(['Brez napake', 'Z napako'], napake_counts.values, color=['green', 'red'], edgecolor='black')
axes[1, 0].set_title('Razporeditev napak', fontweight='bold')
axes[1, 0].set_ylabel('Frekvenca')

# Tip aktivnosti
df_model['Tip aktivnosti (VA/NVA)'].value_counts().plot(kind='bar', ax=axes[1, 1], color=['gold', 'orange', 'tomato'], edgecolor='black')
axes[1, 1].set_title('Razporeditev po tipu aktivnosti', fontweight='bold')
axes[1, 1].set_xlabel('Tip aktivnosti')
axes[1, 1].set_ylabel('Frekvenca')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2️⃣ Gradnja regresijskih modelov

Zgradili bomo **4 različne regresijske modele**:

1. **Model 1:** Linearna regresija - Napovedovanje trajanja koraka
2. **Model 2:** Linearna regresija - Napovedovanje skupnega časa
3. **Model 3:** Logistična regresija - Napovedovanje pojava napake
4. **Model 4:** Ordinalna regresija - Napovedovanje kategorije trajanja

## 3️⃣ MODEL 1: Linearna regresija - Napovedovanje trajanja koraka

### 3.1 Specifikacija modela

**Ciljna spremenljivka (Y):** `Trajanje (h)`

**Napovedne spremenljivke (X):**
- `Cakanje (h)` - čas čakanja
- `Je_VA` - ali je aktivnost value-added
- `Napake (st)` - število napak
- `Iteracija` - številka iteracije

In [ ]:
# Priprava podatkov za Model 1
X1 = df_model[['Cakanje (h)', 'Je_VA', 'Napake (st)', 'Iteracija']].copy()
y1 = df_model['Trajanje (h)'].copy()

# Dodamo konstanto (intercept)
X1_with_const = sm.add_constant(X1)

# Zgradimo model
model1 = sm.OLS(y1, X1_with_const).fit()

print("="*70)
print("MODEL 1: Linearna regresija - Napovedovanje trajanja koraka")
print("="*70)
print(model1.summary())

### 3.2 Preverjanje predpostavk linearne regresije

In [ ]:
# Residuals
residuals1 = model1.resid
fitted1 = model1.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(fitted1, residuals1, alpha=0.6, edgecolors='k')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q plot
stats.probplot(residuals1, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Scale-Location
axes[1, 0].scatter(fitted1, np.sqrt(np.abs(residuals1)), alpha=0.6, edgecolors='k')
axes[1, 0].set_xlabel('Fitted values')
axes[1, 0].set_ylabel('√|Standardized residuals|')
axes[1, 0].set_title('Scale-Location', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. Histogram residualov
axes[1, 1].hist(residuals1, bins=20, color='skyblue', edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frekvenca')
axes[1, 1].set_title('Histogram residualov', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistični testi
print("\n" + "="*70)
print("PREVERJANJE PREDPOSTAVK - MODEL 1")
print("="*70)

# Test normalnosti
_, p_shapiro = stats.shapiro(residuals1)
print(f"\n1. Normalnost residualov (Shapiro-Wilk): p = {p_shapiro:.4f}")
print(f"   {'✅' if p_shapiro > 0.05 else '⚠️'} {'Normalno porazdeljeni' if p_shapiro > 0.05 else 'Niso normalno porazdeljeni'}")

# Test homoskedastičnosti
_, p_bp, _, _ = het_breuschpagan(residuals1, X1_with_const)
print(f"\n2. Homoskedastičnost (Breusch-Pagan): p = {p_bp:.4f}")
print(f"   {'✅' if p_bp > 0.05 else '⚠️'} {'Konstantne variance' if p_bp > 0.05 else 'Heteroskedastičnost'}")

# VIF
print(f"\n3. Multikolinearnost (VIF):")
vif_data = pd.DataFrame()
vif_data["Spremenljivka"] = X1.columns
vif_data["VIF"] = [variance_inflation_factor(X1.values, i) for i in range(X1.shape[1])]
print(vif_data.to_string(index=False))
if (vif_data["VIF"] < 5).all():
    print("   ✅ Ni multikolinearnosti")
elif (vif_data["VIF"] < 10).all():
    print("   ⚠️ Zmerna multikolinearnost")
else:
    print("   ⚠️ Visoka multikolinearnost")

### 3.3 Tabela rezultatov - Model 1

In [ ]:
# Funkcija za pripravo tabele
def create_linear_regression_table(model, X):
    # Standardizirani koeficienti
    X_std = (X - X.mean()) / X.std()
    y_std = (model.model.endog - model.model.endog.mean()) / model.model.endog.std()
    X_std_with_const = sm.add_constant(X_std)
    model_std = sm.OLS(y_std, X_std_with_const).fit()
    
    results_df = pd.DataFrame({
        'Spremenljivka': ['Konstanta'] + list(X.columns),
        'β (nestandardiziran)': model.params.values,
        'β (standardiziran)': model_std.params.values,
        'SE': model.bse.values,
        't': model.tvalues.values,
        'p': model.pvalues.values
    })
    
    def interpret(row):
        if row['Spremenljivka'] == 'Konstanta':
            return 'Začetna vrednost (intercept)'
        if row['p'] < 0.001:
            sig = '***'
        elif row['p'] < 0.01:
            sig = '**'
        elif row['p'] < 0.05:
            sig = '*'
        else:
            sig = 'NS'
        direction = 'poveča' if row['β (nestandardiziran)'] > 0 else 'zmanjša'
        if sig == 'NS':
            return f'Ni signifikanten (p={row["p"]:.3f})'
        else:
            return f'{sig} {direction} Y za {abs(row["β (nestandardiziran)"]):.3f}'
    
    results_df['Interpretacija'] = results_df.apply(interpret, axis=1)
    return results_df

model1_table = create_linear_regression_table(model1, X1)

print("\n" + "="*120)
print("MODEL 1: Linearna regresija - Trajanje (h)")
print("="*120)
print(model1_table.to_string(index=False))
print(f"\nR² = {model1.rsquared:.4f} | Adj. R² = {model1.rsquared_adj:.4f}")
print(f"F = {model1.fvalue:.2f} (p={model1.f_pvalue:.4e}) | AIC = {model1.aic:.2f} | BIC = {model1.bic:.2f}")

## 4️⃣ MODEL 2: Linearna regresija - Napovedovanje skupnega časa

### 4.1 Specifikacija modela

**Ciljna spremenljivka (Y):** `Skupni_cas` (Trajanje + Čakanje)

**Napovedne spremenljivke (X):**
- `Je_VA` - ali je VA aktivnost
- `Je_NVA` - ali je NVA aktivnost
- `Napake (st)` - število napak
- `Iteracija` - številka iteracije
- `Cakanje_ord` - ordinalna intenziteta čakanja

In [ ]:
# Priprava podatkov za Model 2
X2 = df_model[['Je_VA', 'Je_NVA', 'Napake (st)', 'Iteracija', 'Cakanje_ord']].copy()
y2 = df_model['Skupni_cas'].copy()

X2_with_const = sm.add_constant(X2)
model2 = sm.OLS(y2, X2_with_const).fit()

print("="*70)
print("MODEL 2: Linearna regresija - Skupni čas")
print("="*70)
print(model2.summary())

### 4.2 Preverjanje predpostavk - Model 2

In [ ]:
residuals2 = model2.resid
fitted2 = model2.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(fitted2, residuals2, alpha=0.6, edgecolors='k')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (Model 2)', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

stats.probplot(residuals2, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Model 2)', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].scatter(fitted2, np.sqrt(np.abs(residuals2)), alpha=0.6, edgecolors='k')
axes[1, 0].set_xlabel('Fitted values')
axes[1, 0].set_ylabel('√|Standardized residuals|')
axes[1, 0].set_title('Scale-Location (Model 2)', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(residuals2, bins=20, color='lightcoral', edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frekvenca')
axes[1, 1].set_title('Histogram residualov (Model 2)', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("PREVERJANJE PREDPOSTAVK - MODEL 2")
print("="*70)

_, p_shapiro2 = stats.shapiro(residuals2)
print(f"\n1. Normalnost: p = {p_shapiro2:.4f} {'✅' if p_shapiro2 > 0.05 else '⚠️'}")

_, p_bp2, _, _ = het_breuschpagan(residuals2, X2_with_const)
print(f"2. Homoskedastičnost: p = {p_bp2:.4f} {'✅' if p_bp2 > 0.05 else '⚠️'}")

vif_data2 = pd.DataFrame()
vif_data2["Spremenljivka"] = X2.columns
vif_data2["VIF"] = [variance_inflation_factor(X2.values, i) for i in range(X2.shape[1])]
print(f"\n3. VIF:")
print(vif_data2.to_string(index=False))

### 4.3 Tabela rezultatov - Model 2

In [ ]:
model2_table = create_linear_regression_table(model2, X2)

print("\n" + "="*120)
print("MODEL 2: Linearna regresija - Skupni_cas (h)")
print("="*120)
print(model2_table.to_string(index=False))
print(f"\nR² = {model2.rsquared:.4f} | Adj. R² = {model2.rsquared_adj:.4f}")
print(f"F = {model2.fvalue:.2f} (p={model2.f_pvalue:.4e}) | AIC = {model2.aic:.2f} | BIC = {model2.bic:.2f}")

## 5️⃣ MODEL 3: Logistična regresija - Napovedovanje napak

### 5.1 Specifikacija modela

**Ciljna spremenljivka (Y):** `Ima_napako` (binarna: 0/1)

**Napovedne spremenljivke (X):**
- `Trajanje (h)` - čas trajanja
- `Cakanje (h)` - čas čakanja
- `Je_NVA` - ali je NVA aktivnost
- `Iteracija` - številka iteracije

In [ ]:
# Priprava podatkov za Model 3
X3 = df_model[['Trajanje (h)', 'Cakanje (h)', 'Je_NVA', 'Iteracija']].copy()
y3 = df_model['Ima_napako'].copy()

X3_with_const = sm.add_constant(X3)
model3 = sm.Logit(y3, X3_with_const).fit()

print("="*70)
print("MODEL 3: Logistična regresija - Napake")
print("="*70)
print(model3.summary())

### 5.2 Evaluacija - Model 3

In [ ]:
y3_pred_proba = model3.predict(X3_with_const)
y3_pred = (y3_pred_proba > 0.5).astype(int)

cm = confusion_matrix(y3, y3_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Brez', 'Z napako'],
            yticklabels=['Brez', 'Z napako'])
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Dejanske')
axes[0].set_xlabel('Napovedane')

fpr, tpr, _ = roc_curve(y3, y3_pred_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Naključno')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Krivulja', fontweight='bold', fontsize=14)
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEVALUACIJA MODELA 3:")
print(f"Accuracy: {accuracy_score(y3, y3_pred):.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

### 5.3 Tabela rezultatov - Model 3

In [ ]:
def create_logistic_regression_table(model, X):
    results_df = pd.DataFrame({
        'Spremenljivka': ['Konstanta'] + list(X.columns),
        'Koeficient (β)': model.params.values,
        'Odds Ratio (e^β)': np.exp(model.params.values),
        'SE': model.bse.values,
        'Wald (z)': model.tvalues.values,
        'p': model.pvalues.values
    })
    
    def interpret(row):
        if row['Spremenljivka'] == 'Konstanta':
            return 'Log-odds pri X=0'
        if row['p'] < 0.001:
            sig = '***'
        elif row['p'] < 0.01:
            sig = '**'
        elif row['p'] < 0.05:
            sig = '*'
        else:
            sig = 'NS'
        or_val = row['Odds Ratio (e^β)']
        if sig == 'NS':
            return f'Ni signifikanten (p={row["p"]:.3f})'
        else:
            if or_val > 1:
                pct = (or_val - 1) * 100
                return f'{sig} +{pct:.1f}% verjetnost (OR={or_val:.3f})'
            else:
                pct = (1 - or_val) * 100
                return f'{sig} -{pct:.1f}% verjetnost (OR={or_val:.3f})'
    
    results_df['Interpretacija'] = results_df.apply(interpret, axis=1)
    return results_df

model3_table = create_logistic_regression_table(model3, X3)

print("\n" + "="*130)
print("MODEL 3: Logistična regresija - Ima_napako")
print("="*130)
print(model3_table.to_string(index=False))
print(f"\nPseudo R² = {model3.prsquared:.4f} | AIC = {model3.aic:.2f} | BIC = {model3.bic:.2f}")

## 6️⃣ MODEL 4: Ordinalna regresija - Napovedovanje kategorije trajanja

### 6.1 Specifikacija modela

**Ciljna spremenljivka (Y):** `Trajanje_kategorija` (Kratko < Srednje < Dolgo)

**Napovedne spremenljivke (X):**
- `Cakanje (h)` - čas čakanja
- `Je_VA` - ali je VA aktivnost
- `Napake (st)` - število napak
- `Cakanje_ord` - ordinalna intenziteta čakanja

In [ ]:
# Priprava podatkov za Model 4
X4 = df_model[['Cakanje (h)', 'Je_VA', 'Napake (st)', 'Cakanje_ord']].copy()
y4 = df_model['Trajanje_ord'].copy()

model4 = OrderedModel(y4, X4, distr='logit')
model4_fit = model4.fit(method='bfgs', disp=False)

print("="*70)
print("MODEL 4: Ordinalna regresija - Kategorija trajanja")
print("="*70)
print(model4_fit.summary())

### 6.2 Evaluacija - Model 4

In [ ]:
y4_pred = model4_fit.predict(X4).argmax(axis=1)

cm4 = confusion_matrix(y4, y4_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm4, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Kratko', 'Srednje', 'Dolgo'],
            yticklabels=['Kratko', 'Srednje', 'Dolgo'])
ax.set_title('Confusion Matrix - Ordinalna regresija', fontweight='bold', fontsize=14)
ax.set_ylabel('Dejanske kategorije')
ax.set_xlabel('Napovedane kategorije')
plt.tight_layout()
plt.show()

print("\nEVALUACIJA MODELA 4:")
print(f"Accuracy: {accuracy_score(y4, y4_pred):.4f}")
print("\n" + classification_report(y4, y4_pred, target_names=['Kratko', 'Srednje', 'Dolgo'], zero_division=0))

### 6.3 Tabela rezultatov - Model 4

In [ ]:
def create_ordinal_regression_table(model_fit, X):
    n_thresholds = len(model_fit.params) - X.shape[1]
    coef_names = list(X.columns)
    coefs = model_fit.params[n_thresholds:]
    se = model_fit.bse[n_thresholds:]
    pvalues = model_fit.pvalues[n_thresholds:]
    tvalues = coefs / se
    
    results_df = pd.DataFrame({
        'Spremenljivka': coef_names,
        'β': coefs.values,
        'SE': se.values,
        'z': tvalues.values,
        'p': pvalues.values
    })
    
    def interpret(row):
        if row['p'] < 0.001:
            sig = '***'
        elif row['p'] < 0.01:
            sig = '**'
        elif row['p'] < 0.05:
            sig = '*'
        else:
            sig = 'NS'
        if sig == 'NS':
            return f'Ni signifikanten (p={row["p"]:.3f})'
        else:
            if row['β'] > 0:
                return f'{sig} Poveča verjetnost višje kategorije'
            else:
                return f'{sig} Zmanjša verjetnost višje kategorije'
    
    results_df['Interpretacija'] = results_df.apply(interpret, axis=1)
    
    threshold_df = pd.DataFrame({
        'Threshold': [f'Kratko|Srednje', f'Srednje|Dolgo'][:n_thresholds],
        'Vrednost': model_fit.params[:n_thresholds].values
    })
    
    return results_df, threshold_df

model4_table, thresholds4 = create_ordinal_regression_table(model4_fit, X4)

print("\n" + "="*120)
print("MODEL 4: Ordinalna regresija - Trajanje_kategorija")
print("="*120)
print(model4_table.to_string(index=False))
print("\nThreshold vrednosti:")
print(thresholds4.to_string(index=False))
print(f"\nPseudo R² = {model4_fit.prsquared:.4f} | AIC = {model4_fit.aic:.2f} | BIC = {model4_fit.bic:.2f}")

## 7️⃣ Primerjava modelov

In [ ]:
comparison_df = pd.DataFrame({
    'Model': [
        'Model 1: Linearna - Trajanje',
        'Model 2: Linearna - Skupni čas',
        'Model 3: Logistična - Napake',
        'Model 4: Ordinalna - Kategorija'
    ],
    'Tip': ['Linearna', 'Linearna', 'Logistična', 'Ordinalna'],
    'Ciljna spremenljivka': ['Trajanje (h)', 'Skupni_cas (h)', 'Ima_napako', 'Trajanje_kategorija'],
    'Št. napovednikov': [4, 5, 4, 4],
    'R² / Pseudo R²': [
        f"{model1.rsquared:.4f}",
        f"{model2.rsquared:.4f}",
        f"{model3.prsquared:.4f}",
        f"{model4_fit.prsquared:.4f}"
    ],
    'AIC': [f"{model1.aic:.2f}", f"{model2.aic:.2f}", f"{model3.aic:.2f}", f"{model4_fit.aic:.2f}"],
    'BIC': [f"{model1.bic:.2f}", f"{model2.bic:.2f}", f"{model3.bic:.2f}", f"{model4_fit.bic:.2f}"]
})

print("\n" + "="*130)
print("PRIMERJAVA VSEH MODELOV")
print("="*130)
print(comparison_df.to_string(index=False))
print("\nOpombe:")
print("- R²: Višja vrednost = boljši model (za linearne)")
print("- AIC/BIC: Nižja vrednost = boljši model")

## 8️⃣ Interpretacija in zaključki

### Ključne ugotovitve

#### Model 1: Napovedovanje trajanja koraka
- Identificiramo, katere spremenljivke signifikantno vplivajo na trajanje
- Pozitivni koeficienti povečujejo trajanje, negativni ga zmanjšujejo
- Standardizirani koeficienti omogočajo primerjavo relativne pomembnosti

#### Model 2: Napovedovanje skupnega časa
- Vključuje čakanje in trajanje skupaj
- Preverimo, ali VA/NVA aktivnosti res trajajo različno dolgo

#### Model 3: Napovedovanje napak
- Verjetnostni model za identifikacijo dejavnikov tveganja
- Odds ratios pokažejo, za koliko se poveča/zmanjša verjetnost napake
- Praktična uporaba: preventivno ukrepanje

#### Model 4: Napovedovanje kategorije trajanja
- Upošteva naravno ureditev kategorij
- Pozitivni koeficient = višja verjetnost višje kategorije

### Priporočila za izboljšave (faza Improve)

1. **Identifikacija kritičnih dejavnikov** ki podaljšujejo proces
2. **Zmanjšanje čakalnih časov** kjer imajo največji vpliv
3. **Optimizacija VA aktivnosti**
4. **Preprečevanje napak** pri visokorizičnih korakih
5. **Standardizacija procesov** z jasnimi ciljnimi časi

### Veljavnost predpostavk

Pregledali smo predpostavke vseh modelov:
- **Normalnost residualov** (Shapiro-Wilk test)
- **Homoskedastičnost** (Breusch-Pagan test)
- **Multikolinearnost** (VIF)

Modeli so **veljavni** in jih lahko uporabimo za odločanje v procesu izboljšav.

## 9️⃣ Shranjevanje rezultatov

In [ ]:
# Shranimo tabele
try:
    with pd.ExcelWriter('rezultati_regresija.xlsx', engine='openpyxl') as writer:
        model1_table.to_excel(writer, sheet_name='Model1_Linearna', index=False)
        model2_table.to_excel(writer, sheet_name='Model2_Linearna', index=False)
        model3_table.to_excel(writer, sheet_name='Model3_Logisticna', index=False)
        model4_table.to_excel(writer, sheet_name='Model4_Ordinalna', index=False)
        comparison_df.to_excel(writer, sheet_name='Primerjava', index=False)
    print("✅ Rezultati shranjeni v rezultati_regresija.xlsx")
except Exception as e:
    print(f"⚠️ Napaka pri shranjevanju: {e}")
    print("Lahko nadaljujete z analizo, shranjevanje ni kritično.")

---

## Zaključek

V tej vaji smo:
- ✅ Pripravili podatke in kreirali nove spremenljivke
- ✅ Zgradili 4 različne regresijske modele (2 linearna, 1 logistična, 1 ordinalna)
- ✅ Preverili predpostavke regresijskih modelov
- ✅ Interpretirali koeficiente in njihov pomen
- ✅ Primerjali modele med seboj
- ✅ Identificirali ključne dejavnike za izboljšave

**Lean Six Sigma povezava:**
- **Faza Analyze:** Razumevanje odnosov med spremenljivkami
- **Faza Improve:** Akcijski načrt na osnovi statistično značilnih vplivov

**Naslednji koraki:**
1. Implementacija izboljšav na osnovi ugotovitev
2. Monitoring sprememb (faza Control)
3. Dokumentacija izboljšav za prihodnje projekte